# 第11章 見て、動かす: ロボットの方策モデル (Cosmos3-Edge-Policy-DROID)

第8〜10章は「行動を渡すと、未来の映像を返す」でした。ロボットを実際に動かすには逆向き、つまり
**「カメラの映像と作業の指示を渡すと、次にとるべき行動を返す」** 仕組みが必要です。これを**方策 (policy)** と呼びます。

NVIDIA は Cosmos3-Edge を DROID (Franka のアームで集めた大規模な作業データ) で追加学習した
[Cosmos3-Edge-Policy-DROID](https://huggingface.co/nvidia/Cosmos3-Edge-Policy-DROID) を公開しています。
このモデルは、**行動と、その行動をとったときの未来の映像を同時に**出します。行動を決めながら、その結果を頭の中で思い描いているわけです。

1. DROID の 3 台のカメラの映像を 1 枚にまとめる
2. 指示文を変えて、行動と未来の映像を出させる
3. 出てきた行動の数値を見る

**注意**: 実際の Franka を動かすには、行動の数値を DROID の制御の単位に戻して、ロボットの制御プログラムに渡す仕組みが要ります
(公式は RoboLab というシミュレータにつないだサーバ・クライアントの例を出しています)。この章ではそこまではせず、方策が何を出すかを見ることに集中します。

## 準備

**ランタイム**: 「ランタイム → ランタイムのタイプを変更」で **L4** か **A100** の GPU を選んでください (T4 は対象外)。
A100 (40GB 以上) ではモデル全体を GPU に載せ、L4 (24GB) では使う部品だけを GPU に載せる設定 (CPU オフロード) に自動で切り替えます。

**安全フィルタ**: 生成には NVIDIA の安全フィルタ (`cosmos_guardrail`) を使います。使うには Hugging Face で
[nvidia/Cosmos-1.0-Guardrail](https://huggingface.co/nvidia/Cosmos-1.0-Guardrail) の利用申請をして承認を受け、
Colab の左の鍵のマーク (シークレット) に `HF_TOKEN` を登録しておいてください。承認を待たずに試すときは、下の `GUARDRAIL` の印を外します。

In [ ]:
# @title GPU を確かめる
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
if DEVICE == "cuda":
    GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(torch.cuda.get_device_name(0), f"{GPU_GB:.0f} GB")
    assert torch.cuda.get_device_capability()[0] >= 8, "Ampere 以降の GPU (L4 / A100 など) を選んでください。T4 では動きません。"
else:
    GPU_GB = 0
    print(f"CUDA の GPU が見つかりません ({DEVICE} で動かします。とても遅くなります)")

In [ ]:
# @title インストール (数分)
# Cosmos3OmniPipeline は Diffusers の main ブランチに入っている
!pip install -q -U "transformers>=5.18" accelerate av imageio imageio-ffmpeg huggingface_hub
!pip install -q "diffusers @ git+https://github.com/huggingface/diffusers.git" cosmos_guardrail

In [ ]:
# @title 安全フィルタの設定とサンプル素材を取ってくる関数
import os
import urllib.request
from pathlib import Path

GUARDRAIL = True  # @param {type:"boolean"}
try:  # Colab のシークレットに HF_TOKEN があれば使う
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

ASSETS = Path(os.environ.get("COSMOS_ASSETS", "assets"))
ASSETS.mkdir(parents=True, exist_ok=True)
FW = "https://media.githubusercontent.com/media/NVIDIA/cosmos-framework/main/.agents/skills/cookbooks/cosmos3"
HF = "https://huggingface.co/nvidia/Cosmos3-Edge/resolve/main/assets"


def fetch(url):
    """URL の素材を assets/ に落として、そのパスを返す (2 回目からは落とさない)"""
    path = ASSETS / url.rsplit("/", 1)[-1]
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    return str(path)

In [ ]:
# @title 生成モデル (Generator) を読み込む関数
import gc
import json

from diffusers import Cosmos3OmniPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video
from IPython.display import Image as IPImage, Video, display
from PIL import Image, ImageOps


def load_image(path, max_side=10_000):
    """画像を RGB で読む (max_side より大きければ縮める)"""
    img = path if isinstance(path, Image.Image) else ImageOps.exif_transpose(Image.open(path))
    img = img.convert("RGB")
    img.thumbnail((max_side, max_side))
    return img


def load_pipe(flow_shift):
    """Cosmos3OmniPipeline を読み込む (2 回目からは読み込み済みのものを使う)。
    flow_shift は公式の推奨値: 画像→動画は 12.0、行動 (action) を使う生成は 10.0"""
    global pipe
    if "pipe" not in globals():
        pipe = Cosmos3OmniPipeline.from_pretrained(
            "nvidia/Cosmos3-Edge", torch_dtype=torch.bfloat16, enable_safety_checker=GUARDRAIL)
        if DEVICE == "cuda" and GPU_GB < 40:
            pipe.enable_model_cpu_offload()  # L4 など: 使う部品だけ GPU に載せる
        else:
            pipe.to(DEVICE)
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=flow_shift, use_karras_sigmas=False)
    return pipe


def free_gpu():
    """読み込んだモデルを GPU から降ろす"""
    for name in ("pipe", "model", "processor"):
        globals().pop(name, None)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def show_video(path, width=640):
    display(Video(path, embed=True, width=width))

## 方策モデルを読み込む

ベースの Cosmos3-Edge とは別のチェックポイント (同じ 4B) です。第7〜10章と同じ `Cosmos3OmniPipeline` で読み込めます。

In [ ]:
from diffusers import CosmosActionCondition
from PIL import ImageOps
import numpy as np
import matplotlib.pyplot as plt

POLICY_ID = "nvidia/Cosmos3-Edge-Policy-DROID"
policy = Cosmos3OmniPipeline.from_pretrained(POLICY_ID, torch_dtype=torch.bfloat16, enable_safety_checker=GUARDRAIL)
if DEVICE == "cuda" and GPU_GB < 40:
    policy.enable_model_cpu_offload()
else:
    policy.to(DEVICE)
policy.vae.enable_tiling()  # 動画に戻す段階を分割して、メモリを抑える
policy.scheduler = UniPCMultistepScheduler.from_config(policy.scheduler.config, flow_shift=5.0, use_karras_sigmas=False)

# チェックポイントに書かれている方策の設定 (行動のステップ数・カメラの fps・データの種類)
from huggingface_hub import hf_hub_download
policy_cfg = json.loads(Path(hf_hub_download(POLICY_ID, "checkpoint.json")).read_text())["policy"]
print(policy_cfg)

## 1. DROID の 3 台のカメラの映像を 1 枚にまとめる

DROID のデータは、手首のカメラ 1 台と外から見るカメラ 2 台で撮られています。方策モデルには、
**上半分に手首のカメラ、下半分の左右に外のカメラ 2 台**を並べた 640×540 の 1 枚 (`concat_view`) を渡します (公式のノートブックと同じ並べ方)。
素材は公式のクックブックの DROID のサンプルを使います。

In [ ]:
import av

DROID = f"{FW}/generator/action/assets/droid_lerobot_example/videos"
CAMS = {
    "wrist": f"{DROID}/observation.image.wrist_image_left/chunk-000/file-000.mp4",
    "exterior_1": f"{DROID}/observation.image.exterior_image_1_left/chunk-000/file-000.mp4",
    "exterior_2": f"{DROID}/observation.image.exterior_image_2_left/chunk-000/file-000.mp4",
}


def first_frame(url, t=0.0):
    path = ASSETS / ("droid_" + url.split("observation.image.")[1].split("/")[0] + ".mp4")
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    with av.open(str(path)) as c:
        for f in c.decode(video=0):
            if f.time is not None and f.time >= t:
                return f.to_image().convert("RGB")


def concat_view(wrist, ext1, ext2, size=(640, 540)):
    w, h = size
    top = h // 2
    canvas = Image.new("RGB", size)
    canvas.paste(ImageOps.fit(wrist, (w, top), Image.BICUBIC), (0, 0))
    canvas.paste(ImageOps.fit(ext1, (w // 2, h - top), Image.BICUBIC), (0, top))
    canvas.paste(ImageOps.fit(ext2, (w // 2, h - top), Image.BICUBIC), (w // 2, top))
    return canvas


obs = concat_view(*(first_frame(u) for u in CAMS.values()))
obs.save("ch11_observation.png")
display(obs)

## 2. 指示文を変えて、行動と未来の映像を出させる

指示文 (プロンプト) は、作業をそのまま英語で書きます (アップサンプリングしない)。
同じ観測から指示だけを変えると、出てくる行動と未来の映像が変わるかを見比べます。

In [ ]:
def act(observation, instruction, seed=0):
    """観測 (concat_view の画像) と指示から、(未来の映像, 行動) を返す"""
    result = policy(
        prompt=instruction,
        action=CosmosActionCondition(
            mode="policy",
            chunk_size=policy_cfg["action_chunk_size"],
            domain_name=policy_cfg["domain_name"],
            resolution_tier=480,
            image=observation,
            view_point="concat_view",
        ),
        fps=int(policy_cfg["conditioning_fps"]),
        num_inference_steps=30,
        guidance_scale=1.0,
        use_system_prompt=False,
        generator=torch.Generator(device=DEVICE).manual_seed(seed),
    )
    action = result.action[0].float().cpu().numpy() if result.action is not None else None
    return result.video, action


instructions = [
    "Pick up the object and place it in the target container.",
    "Move the gripper to the left side of the table.",
]
plans = {}
for text in instructions:
    video, action = act(obs, text)
    plans[text] = action
    out = f"ch11_policy_{len(plans)}.mp4"
    export_to_video(video, out, fps=int(policy_cfg["conditioning_fps"]), macro_block_size=1)
    print(text, "→ 行動", None if action is None else action.shape)
    show_video(out, 480)

## 3. 出てきた行動の数値を見る

行動は `[ステップ数, 次元]` の配列です。モデルカードには「8 個」とありますが、Diffusers で試したところ 1 ステップあたり **10 個**の数が返ってきました
(第10章の UMI と同じ「手先の位置と向き 9 個 + グリッパー 1 個」の形と考えられます)。値は学習のときに正規化した値で、そのままではメートルや角度の単位ではありません。
指示によって行動の並びがどう変わるかを、次元ごとに描いて比べます。

In [ ]:
fig, axes = plt.subplots(1, len(plans), figsize=(6 * len(plans), 3.2), sharey=True)
axes = np.atleast_1d(axes)
for ax, (text, action) in zip(axes, plans.items()):
    if action is None:
        continue
    for d in range(action.shape[1]):
        ax.plot(action[:, d], label=f"dim {d}")
    ax.set_title(text[:45], fontsize=9)
    ax.set_xlabel("step")
axes[0].set_ylabel("normalized value")
axes[-1].legend(fontsize=7, loc="upper right")
plt.show()

## 方策を実機につなぐには

公式の構成は「方策サーバ」と「ロボット側のクライアント」に分かれています。

1. クライアントがカメラ 3 台の画像とロボットの状態を送る
2. サーバ (このモデル) が 32 ステップぶんの行動を返す
3. クライアントが行動を順に実行し、また 1 に戻る (5〜15Hz)

Jetson AGX Thor では 1 回の推論が 1.5〜2.6 秒ほどで、5Hz の制御に間に合うと報告されています (モデルカードの測定)。
SO-101 のような別のロボットで使うには、そのロボットのデータで追加学習が必要です (Cosmos Framework の方策のポストトレーニングのレシピ)。

## うまくいかないとき

- **`result.action` が None**: 方策のチェックポイント (`Cosmos3-Edge-Policy-DROID`) ではなく、ベースの `Cosmos3-Edge` を読み込んでいないか確認してください
- **GPU メモリが足りない**: 第7〜10章のモデルを読み込んだままなら、ランタイムを再起動してからこのノートブックを実行します
- **映像が観測と関係なくなる**: DROID と違うカメラ配置の画像を入れると、方策は見たことのない状況として扱います。カメラの並べ方 (`concat_view`) は公式と同じにしてください